In [ ]:
# 14_out_of_time_subgroup_liftboot.ipynb
# C-2 out-of-time (train 2020-22 -> test 2023-24), C-3 subgroup fairness by
# income/education quartile, C-5 bootstrap stability of the lift screening boundary.
# Runs locally/cloud; no API key. Writes three CSVs to results/tables/.
import json
from pathlib import Path
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
DATA, MODELS, TAB = ROOT/"data", ROOT/"models", ROOT/"results"/"tables"
SEED = 42
FEATS = ["age","HE_BMI","HE_wc","HE_wt","HE_ht","N_EN","N_CHO","N_SUGAR","N_CHOL",
         "sm_presnt","dr_month","pa_aerobic","L_BR_FQ","edu","ho_incm","D_1_1"]
par = {s: json.load(open(MODELS/f"xgb_{s}_params.json")) for s in ("male","female")}
PAR = {"Male":par["male"], "Female":par["female"]}
d = pd.read_parquet(DATA/"analytic_dataset.parquet")
def fit(train, test, params):
    clf = xgb.XGBClassifier(**params, eval_metric="logloss", random_state=SEED, n_jobs=4)
    clf.fit(train[FEATS], train["diabetes"]); return clf.predict_proba(test[FEATS])[:,1]
# C-2
rows=[]
for sx in ("Male","Female"):
    ds=d[d.sex==sx]; tr=ds[ds.year.isin([2020,2021,2022])]; te=ds[ds.year.isin([2023,2024])]
    p=fit(tr,te,PAR[sx]); y=te.diabetes.values
    rows.append([sx,roc_auc_score(y,p),average_precision_score(y,p),brier_score_loss(y,p),len(tr),len(te),y.mean()])
pd.DataFrame(rows,columns=["Sex","AUC","PR_AUC","Brier","n_train","n_test","prev"]).round(3).to_csv(TAB/"table_c2_out_of_time.csv",index=False)
print("C-2 out-of-time written")
# C-3
sm,sf = pd.read_parquet(DATA/"split_male.parquet"), pd.read_parquet(DATA/"split_female.parquet")
srows=[]
for sx,sdf in [("Male",sm),("Female",sf)]:
    tr=sdf[sdf.split=="train"]; te=sdf[sdf.split=="test"].copy(); te["p"]=fit(tr,te,PAR[sx])
    for var,lab in [("ho_incm","Income"),("edu","Education")]:
        for q in sorted(te[var].dropna().unique()):
            m=te[var]==q; y=te.loc[m,"diabetes"].values; pp=te.loc[m,"p"].values
            auc=roc_auc_score(y,pp) if len(np.unique(y))>1 else np.nan
            srows.append([sx,lab,int(q),len(y),round(float(y.mean()),3),round(float(pp.mean()),3),round(auc,3) if auc==auc else "n/a"])
pd.DataFrame(srows,columns=["Sex","Dimension","Quartile","n","Observed","MeanPred","AUC"]).to_csv(TAB/"table_c3_subgroup_fairness.csv",index=False)
print("C-3 subgroup fairness written")
# C-5
scm,scf = pd.read_parquet(DATA/"scored_male.parquet"), pd.read_parquet(DATA/"scored_female.parquet")
def boot(df,mult=2.0,B=2000,col="prob_cal"):
    t=df[df.split=="test"][[col,"diabetes"]].dropna().values; rng=np.random.default_rng(SEED)
    n=len(t); base=t[:,1].mean(); fr=np.arange(0.05,1.0001,0.05); out=[]
    for _ in range(B):
        s=t[rng.integers(0,n,n)]; ys=s[np.argsort(-s[:,0]),1]; chosen=np.nan
        for f in fr:
            k=max(1,int(round(f*n)))
            if ys[:k].mean()>=mult*base: chosen=f
        out.append(chosen)
    out=np.array([o for o in out if o==o]); return base,np.nanmedian(out),np.nanpercentile(out,2.5),np.nanpercentile(out,97.5)
lrows=[]
for sx,df in [("Male",scm),("Female",scf)]:
    base,med,lo,hi=boot(df); lrows.append([sx,round(base,3),round(med,2),round(lo,2),round(hi,2)]); print(sx,"lift boundary median",med,"95%CI",lo,hi)
pd.DataFrame(lrows,columns=["Sex","Base_prev","Median_topfrac","CI_low","CI_high"]).to_csv(TAB/"table_c5_lift_bootstrap.csv",index=False)
print("C-5 lift bootstrap written")
